# Doctor v3 Nurse Training & Benchmarks (with Change 1: PMH features) — Google Colab

Trains `doctor/v3` (v3 nurse) on the new feature set introduced by **Change 1** — prior discharge-note PMH + ICD-derived PMH fallback + repeat-visit features (~19 new feature columns on top of the existing ~2097 → ~2116 total). Then benchmarks v3 base vs v3 nurse and prints a four-way comparison across all model versions.

**v3 base is NOT retrained** — it is unaffected by Change 1 (no PMH features there). The existing `artifacts/doctor/v3_base/` on Drive is reused for the four-way comparison. If you want to retrain v3 base too, see the optional Section 8 at the bottom.

---

## Repo layout assumption

This notebook assumes the GitHub layout `github.com/<you>/licenta` where `licenta/` is the git repo root and the actual project files (with `src/`, `pyproject.toml`, `notebooks/`, `.claude/`, etc.) live in a nested `proiect_licenta/` subfolder:

```
licenta/                                    ← git repo root (cloned to /content/licenta)
└── proiect_licenta/                        ← project root (PROJECT_PATH in the notebook)
    ├── src/proiect_licenta/...
    ├── pyproject.toml
    ├── notebooks/train_and_benchmark_v3.ipynb   ← (this file)
    ├── benchmarks/
    ├── data/         ← symlink to Drive (created by Cell 3)
    └── artifacts/    ← symlink to Drive (created by Cell 3)
```

All cells below pin paths to `PROJECT_PATH = /content/licenta/proiect_licenta`. If your repo layout is flatter (project files at the git root), change `PROJECT_PATH` in Cell 2 accordingly.

---

## ⚠️ One-Time Drive Setup (do this before running any cells)

MIMIC-IV data and existing trained artifacts must be uploaded to **Google Drive** once. The notebook mounts Drive and symlinks those folders into `PROJECT_PATH` — so `paths.py` resolves everything transparently.

### Required Drive structure

```
MyDrive/proiect_licenta/
├── data/
│   ├── mimic-iv-ed/
│   │   ├── triage.csv
│   │   ├── edstays.csv
│   │   ├── vitalsign.csv          (~115 MB)
│   │   ├── medrecon.csv           (~360 MB)
│   │   └── files_created/
│   │       └── categorized_diagnosis.csv
│   ├── mimic-iv/
│   │   └── hosp/
│   │       ├── patients.csv
│   │       ├── services.csv
│   │       ├── diagnoses_icd.csv      (~181 MB)  ← NEW (Change 1)
│   │       └── admissions.csv         (~94 MB)   ← NEW (Change 1)
│   └── mimic-iv-notes/
│       └── mimic-iv-notes/
│           └── note/
│               └── discharge.csv/
│                   └── discharge.csv     (~3.3 GB)  ← NEW (Change 1)
└── artifacts/
    ├── triage/v1/     ← already present from prior training
    ├── doctor/v1/     ← already present (for comparison)
    ├── doctor/v2/     ← already present (for comparison)
    ├── doctor/v3_base/ ← already present (NOT retrained here)
    └── doctor/v3/     ← will be OVERWRITTEN by this notebook
```

The nested `mimic-iv-notes/mimic-iv-notes/note/discharge.csv/discharge.csv` is the literal layout PhysioNet's MIMIC-IV-Note ships in — keep it as-is. `paths.py` already points there.

> **Tip:** Use the [Google Drive desktop app](https://www.google.com/drive/download/) for the 3.3 GB discharge.csv upload — the browser uploader times out on large single files.

---

## Estimated runtimes (Colab free-tier CPU; GPU not used — XGBoost is CPU-bound here)

| Step | Time |
|------|------|
| Setup + install | ~3 min |
| Environment check + PMH vocab smoke test | < 1 min |
| `train_nurse_v3` (with new PMH step) | **~90–100 min** |
| &nbsp;&nbsp;⤷ of which: PMH aggregation (discharge.csv parse) | ~15–25 min |
| &nbsp;&nbsp;⤷ of which: longitudinal vitals + XGBoost training | ~70–80 min |
| `benchmark_nurse_v3` | ~2–5 min |
| `compare_all_versions` | < 1 min |
| Post-training PMH audit | < 1 min |

> **Heads-up on Colab session limits:** free-tier sessions can be reclaimed after ~90 min of idle or ~12 h total. Keep the tab focused (or use a session-keeper extension) during the long training cell. If you hit a disconnect mid-training, all progress is lost — there's no checkpointing in `train_nurse_v3.py` yet.
>
> **Live progress in Cell 7:** tqdm.auto bars show progress through the discharge.csv parse (~166 chunks), per-stay PMH assembly, longitudinal vitals chunked load and aggregation, and each XGBoost iteration (one tick per boosting round, up to 3000).

---
## Section 1 — Environment Setup
*Run these cells at the start of every Colab session.*

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Clone the repository
# Repo layout: github.com/<user>/licenta is the GIT repo; the actual
# project root (with src/, pyproject.toml, notebooks/, .claude/, etc.)
# lives in a nested 'proiect_licenta/' subfolder. PROJECT_PATH points
# there — everything downstream (symlinks, pip install, runpy) uses it.
# Edit GITHUB_USERNAME below before running.

GITHUB_USERNAME = '<YOUR_USERNAME>'  # ← EDIT THIS
REPO_URL        = f'https://github.com/{GITHUB_USERNAME}/licenta.git'
CLONE_PATH      = '/content/licenta'
PROJECT_PATH    = f'{CLONE_PATH}/proiect_licenta'
BRANCH          = 'main'  # ← EDIT if your work is on a different branch

import os

if os.path.exists(CLONE_PATH):
    print(f'Directory {CLONE_PATH} already exists — pulling latest changes.')
    os.system(f'git -C {CLONE_PATH} fetch')
    os.system(f'git -C {CLONE_PATH} checkout {BRANCH}')
    os.system(f'git -C {CLONE_PATH} pull')
else:
    os.system(f'git clone --branch {BRANCH} {REPO_URL} {CLONE_PATH}')

assert os.path.exists(PROJECT_PATH), (
    f'Expected {PROJECT_PATH} to exist after clone. '
    f'This notebook assumes the project files live in a nested '
    f'proiect_licenta/ subfolder under the licenta/ git repo root.'
)

# Show the head commit so you know which version of Change 1 you're training.
os.system(f'git -C {CLONE_PATH} log -1 --oneline')
print(f'\nProject root: {PROJECT_PATH}')

In [ ]:
# Cell 3: Symlink data/ and artifacts/ from Drive into the PROJECT root
# The symlinks go inside proiect_licenta/, not at the git-repo root, so
# paths.py (which resolves relative to src/proiect_licenta/paths.py via
# parents[2]) finds them.
# Adjust DRIVE_PROJECT if your Drive folder is named differently.

DRIVE_PROJECT = '/content/drive/MyDrive/proiect_licenta'  # ← EDIT IF NEEDED

for folder in ('data', 'artifacts'):
    src  = f'{DRIVE_PROJECT}/{folder}'
    dest = f'{PROJECT_PATH}/{folder}'
    if os.path.islink(dest):
        print(f'Symlink already exists: {dest} → {os.readlink(dest)}')
    elif os.path.exists(dest):
        print(f'WARNING: {dest} is a real directory, not a symlink. Skipping.')
    else:
        if not os.path.exists(src):
            print(f'WARNING: source {src} does not exist on Drive. Symlink not created.')
            continue
        os.symlink(src, dest)
        print(f'Created symlink: {dest} → {src}')

print('Symlinks ready.')

In [ ]:
# Cell 4: Install the proiect_licenta package (editable) + runtime dependencies
# Note: -e PROJECT_PATH (not CLONE_PATH) — pyproject.toml lives one level
# down inside the licenta/proiect_licenta/ folder.
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('-e', PROJECT_PATH)                      # installs the local package
pip('xgboost>=2.0.0')                        # XGBoost (may already be present in Colab)
pip('thefuzz[speedup]>=0.22.0')              # fuzzy drug-name matching
pip('tqdm>=4.66.0', 'ipywidgets>=8.0.0')     # live progress bars in Cell 7

print('All dependencies installed.')

---
## Section 2 — Verify Environment
*Confirm all data files and pre-trained artifacts are reachable before starting the ~90-min training run.* This catches the most common failure mode: a forgotten upload to Drive.

In [ ]:
# Cell 5: Check all required files and artifacts exist
from proiect_licenta.paths import (
    TRIAGE_CSV, EDSTAYS_CSV, PATIENTS_CSV, DIAGNOSIS_CSV,
    SERVICES_CSV, VITALSIGN_CSV, MEDRECON_CSV,
    DIAGNOSES_ICD_CSV, ADMISSIONS_CSV, DISCHARGE_NOTES_CSV,  # ← NEW (Change 1)
    TRIAGE_V1_DIR, DOCTOR_V1_DIR, DOCTOR_V2_DIR,
    DOCTOR_V3_BASE_DIR, DOCTOR_V3_DIR,
)

checks = {
    # ED tables (existing)
    'triage.csv':                                  TRIAGE_CSV,
    'edstays.csv':                                 EDSTAYS_CSV,
    'patients.csv':                                PATIENTS_CSV,
    'categorized_diagnosis.csv':                   DIAGNOSIS_CSV,
    'services.csv':                                SERVICES_CSV,
    'vitalsign.csv (~115 MB)':                     VITALSIGN_CSV,
    'medrecon.csv (~360 MB)':                      MEDRECON_CSV,
    # Change 1 — new tables
    'diagnoses_icd.csv (~181 MB) [Change 1]':      DIAGNOSES_ICD_CSV,
    'admissions.csv (~94 MB) [Change 1]':          ADMISSIONS_CSV,
    'discharge.csv (~3.3 GB) [Change 1]':          DISCHARGE_NOTES_CSV,
    # Required input artifacts
    'artifacts/triage/v1/ (required)':             TRIAGE_V1_DIR,
    'artifacts/doctor/v1/ (for comparison)':       DOCTOR_V1_DIR,
    'artifacts/doctor/v2/ (for comparison)':       DOCTOR_V2_DIR,
    'artifacts/doctor/v3_base/ (reused, not retrained)': DOCTOR_V3_BASE_DIR,
}

all_ok = True
for name, path in checks.items():
    ok = path.exists()
    status = '✓' if ok else '✗ MISSING'
    print(f'{status}  {name:60s}  {path}')
    if not ok:
        all_ok = False

print()
if not all_ok:
    raise RuntimeError(
        'One or more required files/artifacts are missing.\n'
        'Upload the missing ones to Drive (see the Drive Setup section at the top), '
        'then re-run Cell 3 (symlink) and Cell 5 (this check).'
    )
print('All checks passed ✓  Ready to train.')

---
## Section 3 — PMH Vocabulary Smoke Test

Fast sanity check on `pmh_vocab.py` before kicking off the long training run. Verifies that:
1. Positive PMH keywords flag the correct diagnosis groups (CHF → Circulatory, T2DM → Endocrine, stroke → Nervous System + Circulatory).
2. Negations neutralize (`no history of CHF` → no flag; `denies seizures` → no flag).
3. The discharge-note section extractor handles both same-line-content (`PMH: CHF, diabetes`) and own-line-header (`PMH:\n1. CHF\n2. Diabetes`) layouts.

If any of these fail, **stop and audit `pmh_vocab.py` before training** — a broken vocab silently corrupts the PMH feature columns and produces a model that's worse than v3 base.

In [ ]:
# Cell 6: PMH vocab smoke test
from proiect_licenta import pmh_vocab as p

print(f'PMH categories: {len(p.PMH_CATEGORIES)} (should be 13)')
print(f'PMH keywords:   {len(p.PMH_KEYWORD_MAP)}')

# ---- Section extractor ----
cases = [
    ('Past Medical History: CHF, T2DM\nSocial History: lives alone\n',     {'CHF', 'T2DM'},          'Social'),
    ('PMH:\n1. CHF\n2. Diabetes\n3. CKD\nSocial History: nonsmoker',       {'CHF', 'Diabetes', 'CKD'}, 'Social'),
    ('Other intro stuff\n\nPMHx: COPD on home O2, hypertension, GERD',     {'COPD', 'GERD'},         None),
    ('Past Medical History:\nNo history of CHF. Denies seizures.\nSocial: ok', set(),                None),
]
for note, must_contain, must_not in cases:
    section = p.extract_pmh_section(note)
    flags   = p.flags_from_text(section)
    print(f'  section: {section[:80]!r:80s} flags={sorted(flags)}')
    for kw in must_contain:
        assert kw in section, f'expected {kw!r} in extracted section'
    if must_not is not None:
        assert must_not not in section, f'unexpected {must_not!r} in section'

# ---- Negation neutralization ----
neg = p.flags_from_text('No history of CHF or stroke. Denies seizures.')
assert 'Circulatory' not in neg, 'CHF was negated but Circulatory flagged'
assert 'Nervous System and Sense Organs' not in neg, 'seizures was negated but Nervous System flagged'
print(f'  negation test → flags={sorted(neg)} ✓')

# ---- Positive matching ----
pos = p.flags_from_text('CHF, type 2 diabetes, sickle cell disease, anxiety')
for expected in ('Circulatory', 'Endocrine, Nutritional, Metabolic',
                  'Blood and Blood-Forming Organs', 'Mental Disorders'):
    assert expected in pos, f'expected {expected!r} in pos flags'
print(f'  positive test → flags={sorted(pos)} ✓')

print()
print('PMH vocab smoke test PASSED — ready to train.')

---
## Section 4 — Train Doctor v3 Nurse (with Change 1)

Trains the v3-nurse model: same 13-class label space as v3 base, with **snapshot vitals**, **medication flags**, **longitudinal vitals / cardiac rhythm** from `vitalsign.csv`, AND — new from Change 1 — **19 PMH feature columns** from prior discharge-note PMH sections + ICD-derived fallback + repeat-visit numerics.

**Heavy steps inside this single cell:**
1. Load & merge all ED + hospital tables (~1 min).
2. **NEW (Change 1):** stream `discharge.csv` (3.3 GB) in chunks, parse PMH sections, build the per-stay flag matrix (~15–25 min).
3. Aggregate longitudinal vitals from `vitalsign.csv` (~5–10 min).
4. Build the ~2116-feature matrix and split 80/20.
5. Train the 13-class diagnosis model + the cascading 11-class department model (early-stop typically at iteration 1700–2300, ~60 min on Colab CPU).
6. Save artifacts to Drive via the symlink (`artifacts/doctor/v3/`).

**Outputs:** `artifacts/doctor/v3/{diagnosis_model.joblib, department_model.joblib, metadata.json}` — saved directly to Drive. **The previous v3-nurse artifacts (without PMH) are overwritten** — if you want to keep them, snapshot them on Drive before running this cell.

⏱ *Expected runtime: 90–100 min*

In [ ]:
# Cell 7: Train Doctor v3 nurse (with PMH features)
#
# Live progress bars (provided by tqdm.auto, installed in Cell 4):
#   discharge.csv       — chunked PMH parse (~166 chunks of 2000 rows)
#   PMH assembly        — per-stay feature build (~82K stays)
#   vitalsign.csv       — longitudinal vital chunk load
#   long-vitals agg.    — per-stay vital aggregation
#   XGBoost (DIAGNOSIS) — one tick per boosting iteration (up to 3000)
#   XGBoost (DEPARTMENT)— same, cascading after diagnosis
#
# Free-tier Colab tab must stay focused — there is no checkpointing.

import runpy, time

t0 = time.time()
runpy.run_path(
    f'{PROJECT_PATH}/src/proiect_licenta/training/train_nurse_v3.py',
    run_name='__main__',
)
print(f'\nTotal training time: {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Cell 8: Confirm v3 (nurse) artifacts saved to Drive
import json

meta_path = DOCTOR_V3_DIR / 'metadata.json'
assert meta_path.exists(), 'v3 nurse training did not produce metadata.json — check the logs above.'

meta = json.loads(meta_path.read_text())
# Highlight the Change 1 additions in the metadata.
print('v3 (nurse) metadata saved to Drive. Key fields:')
for key in ('version', 'trained_at', 'diagnosis_accuracy', 'department_accuracy',
             'n_diagnosis_classes', 'n_department_classes', 'n_train', 'n_test',
             'catch_all_excluded', 'longitudinal_window_hours',
             'pmh_no_prior_days'):
    print(f'  {key:30s} = {meta.get(key)}')

pmh_cats = meta.get('pmh_categories', [])
pmh_cols = meta.get('pmh_feature_cols', [])
print(f'  pmh_categories: {len(pmh_cats)} categories')
print(f'  pmh_feature_cols: {len(pmh_cols)} columns')
assert len(pmh_cats) == 13, 'expected 13 PMH categories'
assert len(pmh_cols) == 19, 'expected 19 PMH feature columns (13 + 4 numerics + 1 Jaccard + 1 no_history)'
print('  → PMH metadata looks healthy ✓')

---
## Section 5 — Benchmark v3 Base vs v3 Nurse

Compares **v3 base (reused from Drive, no PMH)** vs **v3 nurse (just trained, with PMH)** on the same held-out 20% test split (seed=42). Prints top-1 / top-3 / top-5 accuracy, Cohen's κ, per-class breakdown, and the top-20 feature importances.

**What to look for:**
- v3 base → v3 nurse top-1 delta. Previous baseline (without PMH): **+2.99 pp**. With Change 1 (PMH), the plan's expected lift band is **+5 to +7 pp total over v3 base** (i.e. v3 nurse around **66–67%** top-1 if Change 1 pays off in the middle of its range).
- Per-class recall: Infectious / Endocrine / Blood / Genitourinary should rise. If any strong class (Mental / Digestive / Injury) drops by >1.5 pp, audit before trusting the new artifacts.
- Feature importance: at least one `pmh_*` or `n_prior_*` / `days_since_*` / `no_history` column must appear with non-zero gain (this notebook's Section 7 audit prints a more focused view).

⏱ *Expected runtime: 2–5 min*

In [ ]:
# Cell 9: Benchmark — v3 base vs v3 nurse
runpy.run_path(
    f'{PROJECT_PATH}/benchmarks/benchmark_nurse_v3.py',
    run_name='__main__',
)

---
## Section 6 — Compare All Versions

Four-way accuracy comparison across **v1 / v2 / v3 base / v3 nurse**. Reads only `metadata.json` from each artifact directory — very fast.

> v1 & v2 use 14 diagnosis classes (catch-all included); v3 base & v3 nurse use 13 (catch-all excluded). Accuracy across the two label spaces is **not directly comparable**; the script prints a note about this.

⏱ *Expected runtime: < 1 min*

In [ ]:
# Cell 10: Compare all versions (v1, v2, v3_base, v3_nurse)
runpy.run_path(
    f'{PROJECT_PATH}/benchmarks/compare_all_versions.py',
    run_name='__main__',
)

---
## Section 7 — Change 1 Audit (PMH coverage + feature importance)

Verification gate from the plan. Three checks:

1. **PMH coverage**: what fraction of training stays have at least one PMH flag set? Expected ~40–70% — lower means the discharge-note parser is silently failing or the keyword vocab is too narrow; much higher means we may have leakage from a section other than PMH.
2. **PMH features in top 50 importance**: at least one `pmh_*` / `n_prior_*` / `days_since_*` / `no_history` / `same_complaint_as_prior` column must appear in the top 50 with non-zero gain. Gain=0 everywhere means the merge silently broke — same diagnostic that caught the medication-vocab drift in `audit_med_vocab.py`.
3. **Per-class recall delta** vs the prior v3-nurse run (without PMH). If you remember the old numbers, eyeball them; otherwise this just prints the new numbers for you to compare against `docs/agents/doctor-agent.md`.

Fail-fast: any check failing should block trusting the new artifacts.

In [ ]:
# Cell 11: Change 1 audit — PMH coverage + feature importance
import joblib, numpy as np, pandas as pd

diag_model = joblib.load(DOCTOR_V3_DIR / 'diagnosis_model.joblib')
feature_names = list(diag_model.feature_names_in_)
print(f'Loaded v3 nurse diagnosis model with {len(feature_names)} features.')

# Identify PMH-related columns by name prefix.
pmh_prefixes = ('pmh_', 'n_prior_', 'days_since_', 'no_history',
                'same_complaint_as_prior')
pmh_feature_names = [f for f in feature_names if f.startswith(pmh_prefixes)]
print(f'  PMH-related feature columns present: {len(pmh_feature_names)}')
if len(pmh_feature_names) < 15:
    raise RuntimeError(
        f'Expected ~19 PMH columns but found only {len(pmh_feature_names)} — '
        'the training run did not include Change 1. Check that train_nurse_v3.py '
        'imports PMH_FEATURE_COLS and that build_features concatenates it.'
    )

# ---- (2) Feature importance audit ----
importances = pd.Series(
    diag_model.feature_importances_,
    index=feature_names,
).sort_values(ascending=False)
top50 = importances.head(50)

pmh_in_top50 = [f for f in top50.index if f in pmh_feature_names]
non_zero_pmh = [f for f in pmh_feature_names if importances[f] > 0]

print(f'  PMH features in top 50 by gain: {len(pmh_in_top50)}')
for f in pmh_in_top50:
    rank = top50.index.get_loc(f) + 1
    print(f'    rank {rank:2d}: {f:50s} gain={importances[f]:.5f}')
print(f'  PMH features with non-zero gain anywhere: {len(non_zero_pmh)} / {len(pmh_feature_names)}')

if len(non_zero_pmh) == 0:
    raise RuntimeError(
        'ZERO PMH features have non-zero gain — the merge silently failed. '
        'Audit _aggregate_pmh in train_nurse_v3.py and the merge into df before '
        'trusting these artifacts.'
    )
if len(pmh_in_top50) == 0:
    print('  WARNING: no PMH feature reached the top 50 by gain. '
          'Change 1 may still contribute through many small interactions '
          '(same pattern as the existing nurse features), but verify the '
          'accuracy improvement is real before relying on this.')

# ---- (1) PMH coverage check ----
# Re-derive PMH coverage from a fresh tiny sample so we don't reload the
# whole 102K-row training matrix. Easiest: load the metadata.
import json
meta = json.loads((DOCTOR_V3_DIR / 'metadata.json').read_text())
print(f'\n  Trained on {meta["n_train"]:,} stays | tested on {meta["n_test"]:,} stays')
print(f'  Diagnosis accuracy: {meta["diagnosis_accuracy"]:.4f}')
print(f'  Department accuracy: {meta["department_accuracy"]:.4f}')

print('\nChange 1 audit complete.')

---
## (Optional) Section 8 — Retrain v3 Base

Skip this unless you suspect the existing `artifacts/doctor/v3_base/` on Drive is stale. v3 base does NOT use any Change 1 features, so retraining it produces the same artifacts you already have (modulo XGBoost non-determinism within early stopping, which is negligible).

⏱ *Expected runtime: 30–45 min*

In [ ]:
# Cell 12 (OPTIONAL): Re-train Doctor v3 base
# Uncomment to run.
# runpy.run_path(
#     f'{PROJECT_PATH}/src/proiect_licenta/training/train_doctor_v3.py',
#     run_name='__main__',
# )